- File: `topics/linear_dispersion/linear_dispersion.py`
- Copyright (C) 2026 Hirokazu Maesaka (RIKEN SPring-8 Center)
- SPDX-License-Identifier: GPL-3.0-or-later
- Licensed under GPL-3.0-or-later; see LICENSE. Provided WITHOUT ANY WARRANTY.

# Time evolution of the dispersion function through a drift space, a sector dipole magnet, and a quadrupole magnet

Memo 2026-005

This memo derives the dispersion function $D(s)$ and its slope $D'(s)$ through the three basic elements, starting
from the linear equations of motion (EOM) of [2026-001, Sec. 11]. The dispersion is the response of the transverse
position to the momentum deviation, so it describes an off-momentum particle.

Scope:
- Static magnetic fields only (no RF), so $\delta' = 0$.
- Every element has constant $h$ and $k$ along its length. Fringe fields are ignored (hard-edge model).
- The sector dipole magnet has end faces perpendicular to the reference orbit, so there is no edge focusing.
- The phase-space vector is $\mathbf{u} = (x, p_x, y, p_y, z, \delta)^\mathsf{T}$ with the conventions of
  [2026-001]: $z = s - \beta_0 c\,t$ and $\delta = (E - E_0)/(\beta_0 c P_0)$. To first order $p_x = x'$.

Section 7 extends the result to a particle with a finite betatron amplitude, where the second-order terms of the
transfer map of [2026-004] add an amplitude-dependent dispersion.

Every result is verified with sympy and numerically against the exact EOM.

In [ ]:
import numpy as np
import sympy as sp
from IPython.display import display
from scipy.integrate import solve_ivp

L = sp.Symbol("L", positive=True)
s = sp.Symbol("s", positive=True)
h = sp.Symbol("h", positive=True)
k = sp.Symbol("k", real=True)
g = sp.Symbol("g", positive=True)  # g = sqrt(|k|) of a quadrupole
beta0 = sp.Symbol("beta_0", positive=True)
gamma0 = sp.Symbol("gamma_0", positive=True)
x_, px, y_, py, z_s, d = sp.symbols("x p_x y p_y z delta", real=True)
state = [x_, px, y_, py, z_s, d]
x0, px0, y0, py0, z0, d0 = sp.symbols("x_0 p_{x0} y_0 p_{y0} z_0 delta_0", real=True)
u0 = sp.Matrix([x0, px0, y0, py0, z0, d0])
D0, Dp0 = sp.symbols("D_0 D'_0", real=True)

# Numerical constants of the checks.
BETA0_NUM = 0.9
GAMMA0_NUM = float(1.0 / np.sqrt(1.0 - BETA0_NUM**2))
AMPLITUDE = 1.0e-6
ODE_RTOL = 1.0e-12
ODE_ATOL = 1.0e-18
RESIDUAL_TOLERANCE = 1.0e-10
N_RANDOM_POINTS = 5
RNG_SEED = 20261126
RANDOM_LOW, RANDOM_HIGH = 0.2, 1.5

## 2. Dispersion equation

The linear horizontal EOM of [2026-001, Sec. 11] with $k_s = 0$ is

$$
x'' + (h^2 + k)\,x = h\,\delta .
$$

The momentum deviation $\delta$ is a constant of motion in a static magnetic field, so this is a linear
inhomogeneous equation with a constant right-hand side. Write $x = x_\beta + D\,\delta$, where $x_\beta$ solves the
homogeneous equation. Then

$$
D'' + (h^2 + k)\,D = h .
$$

The dispersion is the solution of this equation with the entrance conditions $D(0) = D_0$ and $D'(0) = D'_0$.
Write $D = D_p + \tilde{D}$, where $D_p$ is a particular solution and $\tilde{D}$ solves the homogeneous equation.
Then $(\tilde{D}, \tilde{D}')$ propagates with the $(x, p_x)$ block of the transfer matrix of [2026-002], and $D_p$
is the dispersion generated by the element itself: $D_p = 0$ for the drift space and the quadrupole, and
$D_p = 1/h = \rho$ for the bend.

Two cases must be distinguished:

- $\delta = 0$: $x = x_\beta$ and the dispersion plays no role. This is the trivial case $x = p_x = y = p_y = 0$.
- $\delta \neq 0$: the extra term $D(s)\,\delta$ is the closed orbit of that momentum. Even with
  $x_0 = p_{x0} = 0$ the particle follows the dispersive orbit, not the reference orbit.

The vertical plane has no inhomogeneous term, so there is no vertical dispersion in this model.

In [ ]:
D_f = sp.Function("D")


def check_dispersion(D_closed: sp.Expr, Dp_closed: sp.Expr, h_value, k_value) -> None:
    """Check that (D, D') solves D'' + (h^2 + k) D = h with D(0) = D_0 and D'(0) = D'_0."""
    assert sp.simplify(sp.diff(D_closed, L) - Dp_closed) == 0
    assert sp.simplify(sp.diff(D_closed, L, 2) + (h_value**2 + k_value) * D_closed - h_value) == 0
    assert sp.simplify(D_closed.subs(L, 0) - D0) == 0
    assert sp.simplify(Dp_closed.subs(L, 0) - Dp0) == 0


display(sp.Eq(sp.Symbol(r"D'' + (h^2+k)D"), h))
display(sp.Eq(sp.Symbol("x"), sp.Symbol("x_beta") + sp.Symbol("D") * d))

## 3. Drift space

With $h = 0$ and $k = 0$ the dispersion equation is $D'' = 0$, so

$$
D(L) = D_0 + L\,D'_0 , \qquad D'(L) = D'_0 .
$$

A drift space does not generate dispersion, but it propagates an existing one. With $x_0 = p_{x0} = 0$ the orbit is
$x(L) = L\,D'_0\,\delta_0$; if in addition $D'_0 = 0$ the particle stays on the reference orbit.

In [ ]:
D_drift = D0 + L * Dp0
Dp_drift = Dp0
check_dispersion(D_drift, Dp_drift, 0, 0)
display(sp.Eq(sp.Symbol("D(L)"), D_drift))
display(sp.Eq(sp.Symbol("D'(L)"), Dp_drift))
print("drift: D = D_0 + L D'_0 and D' = D'_0")

## 4. Sector dipole magnet

With $h \neq 0$ and $k = 0$ the equation is $D'' + h^2 D = h$. The constant particular solution is $D = 1/h$, so

$$
\begin{aligned}
D(L) &= \frac{1}{h} + \Big(D_0 - \frac{1}{h}\Big)\cos(hL) + \frac{D'_0}{h}\sin(hL) , \\
D'(L) &= -h\Big(D_0 - \frac{1}{h}\Big)\sin(hL) + D'_0\cos(hL) .
\end{aligned}
$$

With $\theta = hL$ and $\rho = 1/h$ the first term $\rho(1-\cos\theta)$ is the dispersion generated by the bend
itself. It is the $M_{16}$ element of the transfer matrix of [2026-002, Sec. 5].

In [ ]:
theta = h * L
rho = 1 / h
D_bend = 1 / h + (D0 - 1 / h) * sp.cos(h * L) + Dp0 / h * sp.sin(h * L)
Dp_bend = -h * (D0 - 1 / h) * sp.sin(h * L) + Dp0 * sp.cos(h * L)
check_dispersion(D_bend, Dp_bend, h, 0)
display(sp.Eq(sp.Symbol("D(L)"), D_bend))
display(sp.Eq(sp.Symbol("D'(L)"), Dp_bend))

D_bend_theta = rho * (1 - sp.cos(theta)) + D0 * sp.cos(theta) + rho * Dp0 * sp.sin(theta)
Dp_bend_theta = (1 - D0 / rho) * sp.sin(theta) + Dp0 * sp.cos(theta)
assert sp.simplify(D_bend_theta - D_bend.subs(h, theta / L)) == 0
assert sp.simplify(Dp_bend_theta - Dp_bend.subs(h, theta / L)) == 0
print("bend: D = rho(1-cos theta) + D_0 cos theta + rho D'_0 sin theta")

# The dispersion generated from D_0 = D'_0 = 0 is the M_16 element of the transfer matrix.
D_generated = sp.simplify(D_bend.subs({D0: 0, Dp0: 0}))
display(sp.Eq(sp.Symbol("D_generated"), D_generated))
assert sp.simplify(D_generated - rho * (1 - sp.cos(theta))) == 0
print("the bend generates rho(1-cos theta) from zero")

## 5. Quadrupole magnet

With $h = 0$ the equation is $D'' + k\,D = 0$: a quadrupole does not generate dispersion, it only focuses or
defocuses an existing one. For $k = g^2 > 0$

$$
\begin{aligned}
D(L) &= D_0\cos(gL) + \frac{D'_0}{g}\sin(gL) , \\
D'(L) &= -g\,D_0\sin(gL) + D'_0\cos(gL) ,
\end{aligned}
$$

and for $k = -g^2 < 0$ the trigonometric functions become hyperbolic. These are the $(x, p_x)$ blocks of the
transfer matrix of [2026-002, Sec. 6].

In [ ]:
D_quad_focus = D0 * sp.cos(g * L) + Dp0 / g * sp.sin(g * L)
Dp_quad_focus = -g * D0 * sp.sin(g * L) + Dp0 * sp.cos(g * L)
D_quad_defocus = D0 * sp.cosh(g * L) + Dp0 / g * sp.sinh(g * L)
Dp_quad_defocus = g * D0 * sp.sinh(g * L) + Dp0 * sp.cosh(g * L)
check_dispersion(D_quad_focus, Dp_quad_focus, 0, g**2)
check_dispersion(D_quad_defocus, Dp_quad_defocus, 0, -(g**2))
display(sp.Eq(sp.Symbol("D(L), k>0"), D_quad_focus))
display(sp.Eq(sp.Symbol("D(L), k<0"), D_quad_defocus))
print("quadrupole: cos/sin for k > 0 and cosh/sinh for k < 0")

# Thin-lens limit: L -> 0 with q = kL fixed.
q_thin = sp.Symbol("q", positive=True)
thin_D = sp.limit(sp.simplify(D_quad_focus.subs(g, sp.sqrt(q_thin / L))), L, 0, "+")
thin_Dp = sp.limit(sp.simplify(Dp_quad_focus.subs(g, sp.sqrt(q_thin / L))), L, 0, "+")
display(sp.Eq(sp.Symbol("D_thin"), thin_D))
display(sp.Eq(sp.Symbol("D'_thin"), thin_Dp))
assert sp.simplify(thin_D - D0) == 0
assert sp.simplify(thin_Dp - (Dp0 - q_thin * D0)) == 0
print("thin lens: D -> D_0 and D' -> D'_0 - q D_0")

## 6. Off-momentum particles and the closed orbit

The decomposition $x = x_\beta + D\,\delta$ separates the betatron motion from the dispersive orbit. For
$\delta_0 \neq 0$ the term $D(s)\,\delta_0$ is the closed orbit of that momentum.

The trivial case $x = p_x = y = p_y = 0$ at the entrance deserves a comment. If $\delta_0 = 0$ the particle is the
reference particle. If $\delta_0 \neq 0$, the entrance conditions $x_0 = p_{x0} = 0$ do not keep the particle on the
reference orbit: the dispersion equation still has the inhomogeneous term $h\delta_0$, so the particle leaves the
reference orbit as soon as it enters a bend.

The dispersion is a property of the lattice, not of a single particle. In a ring the periodicity selects the
periodic dispersion; in a transfer line it is fixed by the source.

In [ ]:
# The dispersion column of the transfer matrix of [2026-002] must agree with D(L) from the closed forms.
M_drift = sp.Matrix([[1, L, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 0, 1, L, 0, 0], [0, 0, 0, 1, 0, 0],
                     [0, 0, 0, 0, 1, L / gamma0**2], [0, 0, 0, 0, 0, 1]])
M_bend = sp.Matrix([
    [sp.cos(theta), rho * sp.sin(theta), 0, 0, 0, rho * (1 - sp.cos(theta))],
    [-sp.sin(theta) / rho, sp.cos(theta), 0, 0, 0, sp.sin(theta)],
    [0, 0, 1, L, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [-sp.sin(theta), -rho * (1 - sp.cos(theta)), 0, 0, 1, L / gamma0**2 - rho * (theta - sp.sin(theta))],
    [0, 0, 0, 0, 0, 1],
])
x_block = sp.Matrix([[sp.cos(g * L), sp.sin(g * L) / g], [-g * sp.sin(g * L), sp.cos(g * L)]])
y_block = sp.Matrix([[sp.cosh(g * L), sp.sinh(g * L) / g], [g * sp.sinh(g * L), sp.cosh(g * L)]])
M_quad = sp.diag(x_block, y_block, sp.Matrix([[1, L / gamma0**2], [0, 1]]))

# The dispersion column is the response to delta_0 with x_0 = p_x0 = 0.
for name, M, D_closed in (("drift", M_drift, D_drift), ("bend", M_bend, D_bend_theta),
                          ("quad k>0", M_quad, D_quad_focus)):
    column = sp.simplify(M[0, 5])
    expected = sp.simplify(D_closed.subs({D0: 0, Dp0: 0}))
    print(f"{name:9s} M_16 = {column}")
    assert sp.simplify(column - expected) == 0
print("the dispersion column of the transfer matrix agrees with D(L) from the closed forms")

# The deviation from the particular solution propagates with the same 2x2 block as (x, p_x). The particular
# solution is 0 for the drift space and the quadrupole, and 1/h = rho for the bend.
for name, M, D_closed, Dp_closed, D_particular in (
        ("drift", M_drift, D_drift, Dp_drift, 0),
        ("bend", M_bend, D_bend_theta, Dp_bend_theta, rho),
        ("quad k>0", M_quad, D_quad_focus, Dp_quad_focus, 0)):
    block = M[0:2, 0:2]
    propagated = block * sp.Matrix([D0 - D_particular, Dp0])
    assert sp.simplify(propagated[0] + D_particular - D_closed) == 0
    assert sp.simplify(propagated[1] - Dp_closed) == 0
print("(D - D_particular, D') propagates with the (x, p_x) block of the transfer matrix")

## 7. Dispersion generated by a finite betatron amplitude

The previous sections assume $x_0 = p_{x0} = y_0 = p_{y0} = 0$, so that the only source of the transverse
displacement is $\delta_0$. For a particle with a finite betatron amplitude the second-order terms of the transfer
map also depend on $\delta_0$, and they add a dispersion-like displacement.

Write the map through second order as $\mathbf{P}(\mathbf{u}_0) = \mathbf{M}\mathbf{u}_0 + \mathbf{w}(\mathbf{u}_0)$
with the notation of [2026-004]. Differentiating with respect to $\delta_0$ gives

$$
\frac{\partial\mathbf{P}}{\partial\delta_0} = \mathbf{R}_{:,6} + \mathbf{A}\,\mathbf{u}_0 ,
\qquad
A_{ij} = \frac{\partial^2 w_i}{\partial\delta_0\,\partial u_{0j}} ,
$$

where $\mathbf{R}_{:,6}$ is the sixth column of the linear transfer matrix, that is the dispersion column
$(\ldots, D, \ldots, D', \ldots)^\mathsf{T}$ of [2026-002]. The first term is the linear dispersion of the previous
sections; the second term is the contribution of the finite betatron amplitude. Because $\mathbf{w}$ is quadratic in
$\mathbf{u}_0$, the matrix $\mathbf{A}$ is independent of $\mathbf{u}_0$ and is a property of the element alone.

The matrix $\mathbf{A}$ is the coefficient of $\delta_0 u_{0j}$ in $w_i$, so it is the set of $T_{ij6}$ with the
factor 2 of the symmetric sum removed: $A_{ij} = T_{ij6}$ for $j \neq 6$ and $A_{i6} = 2\,T_{i66}$. The factor 2
appears because $\partial^2(\delta_0^2)/\partial\delta_0^2 = 2$.

In [ ]:
H3 = (h * x_ - d) * (px**2 + py**2) / 2 + (h * x_ - d) * d**2 / (2 * gamma0**2)
f2_state = sp.Matrix([sp.diff(H3, px), -sp.diff(H3, x_), sp.diff(H3, py), -sp.diff(H3, y_),
                      sp.diff(H3, d), -sp.diff(H3, z_s)])


def second_order_part(M_of, h_value, length=L) -> sp.Matrix:
    """Second-order part w(L) from Duhamel's formula for the linear matrix function M_of(l) and the value of h."""
    f2_h = f2_state.subs(h, h_value)
    u_s = M_of(s) * u0
    f2_s = f2_h.subs(dict(zip(state, u_s)), simultaneous=True)
    integrand = M_of(length - s) * f2_s
    w = sp.zeros(6, 1)
    for i in range(6):
        expanded = sp.expand(integrand[i].rewrite(sp.exp))
        poly = sp.Poly(expanded, *u0)
        for monomial, coeff in poly.terms():
            integral = sp.integrate(sp.expand(coeff), (s, 0, length))
            integral = sp.simplify(sp.expand(integral).rewrite(sp.cos))
            w[i] += integral * sp.prod([u**p for u, p in zip(u0, monomial)])
    return w


def amplitude_matrix(w: sp.Matrix) -> sp.Matrix:
    """A_ij = d^2 w_i / (d delta_0 d u_0j)."""
    return sp.Matrix(6, 6, lambda i, j: sp.simplify(sp.diff(w[i], d0, u0[j])))


M_quad_focus = M_quad

w_drift = second_order_part(lambda length: M_drift.subs(L, length), 0)
w_bend = second_order_part(lambda length: M_bend.subs(L, length), h)
w_quad = second_order_part(lambda length: M_quad_focus.subs(L, length), 0)

A_drift = amplitude_matrix(w_drift)
A_bend = amplitude_matrix(w_bend)
A_quad = amplitude_matrix(w_quad)

for name, A in (("drift", A_drift), ("bend", A_bend), ("quad", A_quad)):
    print(f"A_{name} u_0 =")
    display(sp.simplify(A * u0))

**Drift space.** Only $w_1 = -L\delta_0 p_{x0}$, $w_3 = -L\delta_0 p_{y0}$, and
$w_5 = -L(p_{x0}^2+p_{y0}^2)/2 - 3L\delta_0^2/(2\gamma_0^2)$ contribute, so

$$
\mathbf{A}_\mathrm{drift}\,\mathbf{u}_0
= \big(-L\,p_{x0},\ 0,\ -L\,p_{y0},\ 0,\ -3L\,\delta_0/\gamma_0^2,\ 0\big)^\mathsf{T} .
$$

The drift space therefore does generate a dispersion-like term, but only from the betatron momenta $p_{x0}$ and
$p_{y0}$ and from $\delta_0$ itself; there is no term in $x_0$. With $p_{x0} = p_{y0} = 0$ it vanishes, which is the
result of Section 3. The physical origin is the momentum dependence of the effective drift length: a particle with a
finite slope travels a longer path, so its displacement grows with $\delta_0$.

In [ ]:
A_drift_expected = sp.Matrix([-L * px0, 0, -L * py0, 0, -3 * L * d0 / gamma0**2, 0])
assert sp.simplify(A_drift * u0 - A_drift_expected) == sp.zeros(6, 1)
print("drift: A u_0 = (-L p_x0, 0, -L p_y0, 0, -3L delta_0/gamma_0^2, 0)")

**Sector dipole magnet.** With $\theta = hL$ and $\rho = 1/h$,

$$
\begin{aligned}
\big(\mathbf{A}_\mathrm{bend}\mathbf{u}_0\big)_1 &= \sin^2\theta\,x_0
  - \frac{\sin 2\theta}{2h}\,p_{x0}
  - \Big[\frac{\sin^2\theta}{h} + \frac{1-\cos\theta}{\gamma_0^2 h}\Big]\delta_0 , \\
\big(\mathbf{A}_\mathrm{bend}\mathbf{u}_0\big)_2 &= -\frac{\sin\theta}{\gamma_0^2}\,\delta_0 , \\
\big(\mathbf{A}_\mathrm{bend}\mathbf{u}_0\big)_3 &= -\frac{\sin\theta}{h}\,p_{y0} , \\
\big(\mathbf{A}_\mathrm{bend}\mathbf{u}_0\big)_5 &= \frac{\sin\theta}{\gamma_0^2}\,x_0
  + \frac{1-\cos\theta}{\gamma_0^2 h}\,p_{x0}
  - \frac{3\sin\theta}{\gamma_0^2 h}\,\delta_0 .
\end{aligned}
$$

The terms in $x_0$ and $p_{x0}$ are the new contribution: a particle with a finite betatron amplitude in a bend
acquires an additional momentum-dependent displacement, which is the chromatic effect of the bend on the betatron
orbit. The term in $p_{y0}$ is the coupling of the vertical slope to the horizontal displacement through the bend.

In [ ]:
A_bend_expected = sp.Matrix([
    sp.sin(theta)**2 * x0 - sp.sin(2 * theta) / (2 * h) * px0
    - (sp.sin(theta)**2 / h + (1 - sp.cos(theta)) / (gamma0**2 * h)) * d0,
    -sp.sin(theta) / gamma0**2 * d0,
    -sp.sin(theta) / h * py0,
    0,
    sp.sin(theta) / gamma0**2 * x0 + (1 - sp.cos(theta)) / (gamma0**2 * h) * px0
    - 3 * sp.sin(theta) / (gamma0**2 * h) * d0,
    0,
])
assert sp.simplify(A_bend * u0 - A_bend_expected) == sp.zeros(6, 1)
print("bend: A u_0 agrees with the closed form")

**Quadrupole magnet.** With $\varphi = gL$,

$$
\begin{aligned}
\big(\mathbf{A}_\mathrm{quad}\mathbf{u}_0\big)_1 &= \frac{gL\sin\varphi}{2}\,x_0
  - \frac{\varphi\cos\varphi + \sin\varphi}{2g}\,p_{x0} , \\
\big(\mathbf{A}_\mathrm{quad}\mathbf{u}_0\big)_2 &= \frac{g\,(\varphi\cos\varphi - \sin\varphi)}{2}\,x_0
  + \frac{gL\sin\varphi}{2}\,p_{x0} , \\
\big(\mathbf{A}_\mathrm{quad}\mathbf{u}_0\big)_3 &= -\frac{gL\sinh\varphi}{2}\,y_0
  - \frac{\varphi\cosh\varphi + \sinh\varphi}{2g}\,p_{y0} , \\
\big(\mathbf{A}_\mathrm{quad}\mathbf{u}_0\big)_4 &= -\frac{g\,(\varphi\cosh\varphi - \sinh\varphi)}{2}\,y_0
  - \frac{gL\sinh\varphi}{2}\,p_{y0} , \\
\big(\mathbf{A}_\mathrm{quad}\mathbf{u}_0\big)_5 &= -\frac{3L}{\gamma_0^2}\,\delta_0 .
\end{aligned}
$$

A quadrupole therefore also generates a dispersion-like term, proportional to the betatron amplitude and to
$\delta_0$. It vanishes for $x_0 = p_{x0} = y_0 = p_{y0} = 0$, which is why Section 5 found no dispersion
generation.

In [ ]:
phi = g * L
A_quad_expected = sp.Matrix([
    g * L * sp.sin(phi) / 2 * x0 - (phi * sp.cos(phi) + sp.sin(phi)) / (2 * g) * px0,
    g * (phi * sp.cos(phi) - sp.sin(phi)) / 2 * x0 + g * L * sp.sin(phi) / 2 * px0,
    -g * L * sp.sinh(phi) / 2 * y0 - (phi * sp.cosh(phi) + sp.sinh(phi)) / (2 * g) * py0,
    -g * (phi * sp.cosh(phi) - sp.sinh(phi)) / 2 * y0 - g * L * sp.sinh(phi) / 2 * py0,
    -3 * L / gamma0**2 * d0,
    0,
])
assert sp.simplify(A_quad * u0 - A_quad_expected) == sp.zeros(6, 1)
print("quad: A u_0 agrees with the closed form")

**Interpretation.** The full dependence of the transverse position on $\delta_0$ is

$$
\mathbf{P}(\mathbf{u}_0) = \mathbf{P}\big|_{\delta_0 = 0}
+ \Big[\mathbf{R}_{:,6} + \mathbf{A}\,\mathbf{u}_0\Big]\delta_0 + \mathcal{O}(\delta_0^2) ,
$$

so the effective dispersion seen by a particle of finite amplitude is
$D_\mathrm{eff} = D + (\mathbf{A}\mathbf{u}_0)_1$ in the horizontal plane. The amplitude-dependent part is of second
order in the betatron amplitude, so it is small for a well-behaved beam, but it is the origin of the
amplitude-dependent dispersion and of the chromaticity of the betatron motion.

The limits are consistent: $\mathbf{A}_\mathrm{bend} \to \mathbf{A}_\mathrm{drift}$ for $h \to 0$ and
$\mathbf{A}_\mathrm{quad} \to \mathbf{A}_\mathrm{drift}$ for $g \to 0$.

In [ ]:
# The derivative of the map with respect to delta_0 is the dispersion column plus A u_0.
for name, M, w, A in (("drift", M_drift, w_drift, A_drift), ("bend", M_bend, w_bend, A_bend),
                      ("quad", M_quad_focus, w_quad, A_quad)):
    P = M * u0 + w
    dP = sp.Matrix(6, 1, lambda i, _j: sp.simplify(sp.diff(P[i], d0)))
    residual = sp.simplify(dP - (M[:, 5] + A * u0))
    print(f"{name:6s} dP/ddelta_0 - (R_:,6 + A u_0): {max(abs(e) for e in residual)}")
    assert residual == sp.zeros(6, 1)
print("dP/ddelta_0 = R_:,6 + A u_0 for every element")

# A_ij = T_ij6 for j != 6 and A_i6 = 2 T_i66.
for name, w, A in (("drift", w_drift, A_drift), ("bend", w_bend, A_bend), ("quad", w_quad, A_quad)):
    for i in range(6):
        poly = sp.Poly(sp.expand(w[i]), *u0)
        for j in range(6):
            if j == 5:
                expected = 2 * poly.coeff_monomial(d0**2)
            else:
                expected = poly.coeff_monomial(u0[j] * d0)
            assert sp.simplify(A[i, j] - expected) == 0
print("A_ij = T_ij6 (j != 6) and A_i6 = 2 T_i66")

# Limits: the bend and the quadrupole reduce to the drift space.
assert sp.simplify(A_bend.applyfunc(lambda e: sp.limit(e, h, 0)) - A_drift) == sp.zeros(6, 6)
assert sp.simplify(A_quad.applyfunc(lambda e: sp.limit(e, g, 0)) - A_drift) == sp.zeros(6, 6)
print("h -> 0 and g -> 0 give A_drift")

## 8. Checks

1. Each closed form satisfies the dispersion equation and the entrance conditions.
2. The homogeneous part $(\tilde{D}, \tilde{D}')$ propagates with the $(x, p_x)$ block of the transfer matrix of
   [2026-002], with $D_p = 0$ for the drift space and the quadrupole and $D_p = \rho$ for the bend.
3. The dispersion column of the transfer matrix agrees with $D(L)$ from the closed forms.
4. The exact nonlinear EOM reproduces $x(L) = D(L)\,\delta_0$ for an off-momentum particle at small amplitude.
5. Limits: $h \to 0$ and $k \to 0$ give the drift space.
6. Composition: $D(L_1 + L_2)$ equals the result of propagating through $L_1$ and then $L_2$.
7. $\partial\mathbf{P}/\partial\delta_0 = \mathbf{R}_{:,6} + \mathbf{A}\mathbf{u}_0$ for every element, with
   $A_{ij} = T_{ij6}$ for $j \neq 6$ and $A_{i6} = 2\,T_{i66}$.
8. $\mathbf{A}_\mathrm{bend} \to \mathbf{A}_\mathrm{drift}$ for $h \to 0$ and
   $\mathbf{A}_\mathrm{quad} \to \mathbf{A}_\mathrm{drift}$ for $g \to 0$.

In [ ]:
rng = np.random.default_rng(RNG_SEED)


def random_point(symbols: list) -> dict:
    """Random positive values for the parameters and random real values for the entrance dispersion."""
    values = {}
    for sym in symbols:
        if sym in (D0, Dp0):
            values[sym] = rng.uniform(-1.0, 1.0)
        else:
            values[sym] = rng.uniform(RANDOM_LOW, RANDOM_HIGH)
    return values


def max_residual(expr) -> float:
    """Largest absolute value of an expression over random points."""
    symbols = sorted(sp.sympify(expr).free_symbols, key=str)
    func = sp.lambdify(symbols, expr, "numpy")
    worst = 0.0
    for _ in range(N_RANDOM_POINTS):
        point = random_point(symbols)
        value = np.array(func(*[point[sym] for sym in symbols]), dtype=float)
        worst = max(worst, float(np.max(np.abs(value))))
    return worst


# Check 1: the dispersion equation and the entrance conditions.
ELEMENTS = {
    "drift": (0, 0, D_drift, Dp_drift),
    "bend": (h, 0, D_bend, Dp_bend),
    "quad k>0": (0, g**2, D_quad_focus, Dp_quad_focus),
    "quad k<0": (0, -(g**2), D_quad_defocus, Dp_quad_defocus),
}
for name, (h_value, k_value, D_closed, Dp_closed) in ELEMENTS.items():
    ode_residual = max_residual(sp.diff(D_closed, L, 2) + (h_value**2 + k_value) * D_closed - h_value)
    entrance_D = max_residual(D_closed.subs(L, 0) - D0)
    entrance_Dp = max_residual(Dp_closed.subs(L, 0) - Dp0)
    print(f"{name:9s} ODE residual: {ode_residual:.1e}, D(0): {entrance_D:.1e}, D'(0): {entrance_Dp:.1e}")
    assert ode_residual < RESIDUAL_TOLERANCE
    assert entrance_D < RESIDUAL_TOLERANCE and entrance_Dp < RESIDUAL_TOLERANCE

In [ ]:
# Check 5: limits.
assert sp.simplify(sp.limit(D_bend, h, 0) - D_drift) == 0
assert sp.simplify(sp.limit(Dp_bend, h, 0) - Dp_drift) == 0
assert sp.simplify(sp.limit(D_quad_focus, g, 0) - D_drift) == 0
assert sp.simplify(sp.limit(Dp_quad_focus, g, 0) - Dp_drift) == 0
print("h -> 0 and k -> 0 give the drift space")

# Check 6: composition.
L1, L2 = sp.symbols("L_1 L_2", positive=True)
for name, (h_value, k_value, D_closed, Dp_closed) in ELEMENTS.items():
    lhs = D_closed.subs(L, L1 + L2)
    D_mid = D_closed.subs(L, L1)
    Dp_mid = Dp_closed.subs(L, L1)
    rhs = D_closed.subs(L, L2).subs({D0: D_mid, Dp0: Dp_mid}, simultaneous=True)
    residual = max_residual(lhs - rhs)
    print(f"{name:9s} composition: {residual:.1e}")
    assert residual < RESIDUAL_TOLERANCE

In [ ]:
# Check 4: the exact nonlinear EOM of [2026-001, Sec. 10] with k_s = 0.
R_exact = sp.sqrt(1 + 2 * d + (1 - 1 / gamma0**2) * d**2 - px**2 - py**2)
H_exact = -(1 + h * x_) * R_exact + d + h * (x_ + h * x_**2 / 2) + k * (x_**2 - y_**2) / 2
rhs_exact = [sp.diff(H_exact, px), -sp.diff(H_exact, x_), sp.diff(H_exact, py), -sp.diff(H_exact, y_),
             sp.diff(H_exact, d), -sp.diff(H_exact, z_s)]
rhs_exact_func = sp.lambdify([*state, h, k, gamma0], rhs_exact, "numpy")

NUMERIC_CASES = {
    # name: (h, k, length)
    "drift": (0.0, 0.0, 1.3),
    "bend": (0.2, 0.0, 1.5),
    "quad k>0": (0.0, 2.0, 0.4),
    "quad k<0": (0.0, -2.0, 0.4),
}
DELTA_NUM = 1.0e-3

for case, (h_num, k_num, length) in NUMERIC_CASES.items():
    h_value, k_value, D_closed, _Dp_closed = ELEMENTS[case]
    # Off-momentum particle with x_0 = p_x0 = 0 and a nonzero dispersion at the entrance.
    D0_num, Dp0_num = 0.3, -0.2
    u_start = np.array([0.0, 0.0, 0.0, 0.0, 0.0, DELTA_NUM])
    sol = solve_ivp(lambda _s, u: rhs_exact_func(*u, h_num, k_num, GAMMA0_NUM), (0.0, length), u_start,
                    rtol=ODE_RTOL, atol=ODE_ATOL, method="DOP853")
    x_exact = sol.y[0, -1]
    D_num = float(D_closed.subs({h: h_num, g: float(np.sqrt(abs(k_num))), L: length,
                                 D0: D0_num, Dp0: Dp0_num, gamma0: GAMMA0_NUM}).evalf())
    # The exact orbit starts on the reference orbit, so the dispersive part is D(L) delta_0 with D_0 = D'_0 = 0.
    D_from_zero = float(D_closed.subs({h: h_num, g: float(np.sqrt(abs(k_num))), L: length,
                                       D0: 0, Dp0: 0, gamma0: GAMMA0_NUM}).evalf())
    predicted = D_from_zero * DELTA_NUM
    print(f"{case:9s} exact x(L) = {x_exact:.6e}, D(L) delta_0 = {predicted:.6e}, "
          f"difference = {abs(x_exact - predicted):.1e}")
    # The neglected terms are of second order in delta_0, so the tolerance scales as DELTA_NUM**2.
    assert abs(x_exact - predicted) < 10.0 * DELTA_NUM**2

## 9. Summary and open points

1. The dispersion solves $D'' + (h^2+k)\,D = h$ with the entrance conditions $D_0$, $D'_0$. Its homogeneous part
   propagates with the $(x, p_x)$ block of the transfer matrix, and the particular solution $D_p$ is the dispersion
   generated by the element.
2. Drift space: $D = D_0 + L D'_0$, $D' = D'_0$. No dispersion is generated.
3. Sector dipole magnet: $D = \rho(1-\cos\theta) + D_0\cos\theta + \rho D'_0\sin\theta$. The bend generates
   $\rho(1-\cos\theta)$ from zero.
4. Quadrupole magnet: $D = D_0\cos(gL) + (D'_0/g)\sin(gL)$ for $k > 0$ and the hyperbolic form for $k < 0$. No
   dispersion is generated.
5. Off-momentum particles: $x = x_\beta + D\,\delta$. With $x_0 = p_{x0} = 0$ and $\delta_0 \neq 0$ the particle
   follows the dispersive orbit $D(s)\,\delta_0$, not the reference orbit.
6. Finite betatron amplitude: $\partial\mathbf{P}/\partial\delta_0 = \mathbf{R}_{:,6} + \mathbf{A}\mathbf{u}_0$,
   with $A_{ij} = \partial^2 w_i/\partial\delta_0\partial u_{0j}$. The drift space contributes only through
   $p_{x0}$, $p_{y0}$, and $\delta_0$; the bend and the quadrupole contribute through the betatron amplitude as
   well. The effective dispersion is $D_\mathrm{eff} = D + (\mathbf{A}\mathbf{u}_0)_1$.

Open points (not covered here):

- The periodic dispersion of a ring and the momentum compaction factor.
- The vertical dispersion from skew quadrupoles and solenoids.
- The dispersion in the presence of sextupoles, where it becomes amplitude dependent.
- The second-order correction to the dispersion.

## License

The narrative text in this notebook is licensed under
[CC BY 4.0](../../LICENSES/CC-BY-4.0.txt). Code is licensed under GPL-3.0-or-later.
See [LICENSING.md](../../LICENSING.md).